In [2]:
using JuMP, HiGHS, REopt

In [1]:
using CSV, DataFrames

In [5]:
m = Model(HiGHS.Optimizer)

A JuMP Model
├ solver: HiGHS
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

In [72]:
# Load CSV data
load_data = CSV.read("REopt_Load_Profile.csv", DataFrame)[!, "Electric Load (kW)"]

8760-element Vector{Int64}:
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 ⋮
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0

In [74]:


# Define inputs as a standard Julia Dictionary
inputs_dict = Dict(
    "Site" => Dict("latitude" => 10.80437,
        "longitude" => 106.71927,
        "land_acres" => 0.0,# in hec
        #"min_resil_time_steps" => ,
        "roof_squarefeet" => 2200, #in ft^2 => ~200 m^2
        "sector" => "commercial/industrial",
        #"renewable_electricity_min_fraction" => 0.10,
        #"renewable_electricity_max_fraction" => 0.25,
        ),
    
   # ElectricLoad how to simulate
    # "ElectricLoad" => Dict("doe_reference_name" => "MediumOffice", 
     #   "annual_kwh" => 100000.0,
      #  "normalize_and_scale_load_profile_input" => true
    #),

    #ElectricLoad from CSV file
    #"ElectricLoad" => Dict("path_to_csv" => "REopt_Load_Profile.csv", 
    #    "year" => 2026
    #),

    #ElectricLoad from list
    "ElectricLoad" => Dict("loads_kw" => load_data, 
        "year" => 2026
    ),
    
    "ElectricTariff" => Dict(
        # Average cost per kWh for each month (Jan to Dec)
        "monthly_energy_rates" => [0.12, 0.12, 0.12, 0.14, 0.14, 0.16, 0.16, 0.16, 0.14, 0.12, 0.12, 0.12],
        
        # Peak demand charge per kW for each month
        "monthly_demand_rates" => [15.0, 15.0, 15.0, 20.0, 20.0, 25.0, 25.0, 25.0, 20.0, 15.0, 15.0, 15.0],
        
    ),
    
    "PV" => Dict(
        "size_class" => 2,          # Small commercial 
        "module_type" => 1,         # Premium
        "location" => "roof",
        "array_type" => 1,          # Rooftop, Fixed
        "tilt" => 10.8,             # Optimized for HCMC latitude (~10.8° N)
        "azimuth" => 180.0,         # Facing South
        "losses" => 0.17,            # Adjusted for tropical heat and humidity
        "dc_ac_ratio" => 1.2,

        
        # Financials (Using Euros)
        "installed_cost_per_kw" => 600.0, 
        "om_cost_per_kw" => 8.0,
        
        # Disable Grid Trading
        "can_net_meter" => false,
        "can_wholesale" => false,
        "can_export_beyond_nem_limit" => false,

        # Tax Treatment
        "macrs_option_years" => 0,
        "macrs_bonus_fraction" => 0.0
    ),
    
    "ElectricStorage" => Dict(
        "min_kwh" => 20.0)
)

Dict{String, Dict{String}} with 5 entries:
  "ElectricStorage" => Dict("min_kwh"=>20.0)
  "ElectricTariff"  => Dict("monthly_demand_rates"=>[15.0, 15.0, 15.0, 20.0, 20…
  "Site"            => Dict{String, Any}("latitude"=>10.8044, "longitude"=>106.…
  "ElectricLoad"    => Dict{String, Any}("loads_kw"=>[58.909, 53.747, 65.448, 5…
  "PV"              => Dict{String, Any}("array_type"=>1, "module_type"=>1, "ti…

In [75]:
s = Scenario(inputs_dict)

┌ REopt | Warn: Could not look up EASIUR health costs from point (10.80437,106.71927). Location is likely invalid or outside the CAMx grid.
└ @ REopt C:\Users\hauti\.julia\packages\REopt\8hDDW\src\core\financial.jl:255
┌ REopt | Warn: Could not look up EASIUR health costs from point (10.80437,106.71927). Location is likely invalid or outside the CAMx grid.
└ @ REopt C:\Users\hauti\.julia\packages\REopt\8hDDW\src\core\financial.jl:255
┌ REopt | Warn: Could not look up EASIUR health cost escalation rates from point (10.80437,106.71927). Location is likely invalid or outside the CAMx grid
└ @ REopt C:\Users\hauti\.julia\packages\REopt\8hDDW\src\core\financial.jl:285
┌ REopt | Warn: Could not find AVERT region containing site latitude/longitude. Checking site proximity to AVERT regions.
└ @ REopt C:\Users\hauti\.julia\packages\REopt\8hDDW\src\core\electric_utility.jl:411
┌ REopt | Warn: Your site location (10.80437, 106.71927) is more than 5 miles from the nearest AVERT region. Cannot calc

Scenario(REopt.Settings(1, true, false, false, false, "HiGHS", false), REopt.Site(10.80437, 106.71927, 0.0, 2200, 0, true, "commercial/industrial", "", "", nothing, nothing, nothing, nothing, nothing, nothing, false, true, true, nothing, 1), REopt.PV[REopt.PV(1, 10.8, 1, 0.17, 180.0, 0.4, 0, "PV", "roof", 0, 0, 1.0e9, 600.0, 8.0, 0.005, 0, 0.0, 0.5, 0.01, 0.006, 0.96, 1.2, nothing, 0.3, 0.0, 0.0, 1.0e10, 0.0, 1.0e10, 0.0, 1.0e10, 0.0, 1.0e10, 0.0, 1.0e9, 1, 1.0e9, false, false, false, true, 0.0, 2, Float64[], false, 500000.5380000002, 0.0, 2200)], REopt.Wind(0.0, 0, 7692.0, 42.0, nothing, "residential", 20, Float64[], Float64[], Float64[], Float64[], 0.03, 5, 1.0, 0.5, 0.3, 0.0, 0.0, 1.0e10, 0.0, 1.0e10, 0.0, 1.0e10, 0.0, 1.0e10, 0.0, 1.0e9, 1, 1.0e9, true, true, true, true, 0.0), REopt.Storage(REopt.StorageTypes(["ElectricStorage"], ["ElectricStorage"], String[], String[], String[]), Dict{String, REopt.AbstractStorage}("ElectricStorage" => REopt.ElectricStorage(0.0, 10000.0, 20.0, 1.0

In [76]:
results = run_reopt(m, inputs_dict)

┌ REopt | Warn: Could not look up EASIUR health costs from point (10.80437,106.71927). Location is likely invalid or outside the CAMx grid.
└ @ REopt C:\Users\hauti\.julia\packages\REopt\8hDDW\src\core\financial.jl:255
┌ REopt | Warn: Could not look up EASIUR health costs from point (10.80437,106.71927). Location is likely invalid or outside the CAMx grid.
└ @ REopt C:\Users\hauti\.julia\packages\REopt\8hDDW\src\core\financial.jl:255
┌ REopt | Warn: Could not look up EASIUR health cost escalation rates from point (10.80437,106.71927). Location is likely invalid or outside the CAMx grid
└ @ REopt C:\Users\hauti\.julia\packages\REopt\8hDDW\src\core\financial.jl:285
┌ REopt | Warn: Could not find AVERT region containing site latitude/longitude. Checking site proximity to AVERT regions.
└ @ REopt C:\Users\hauti\.julia\packages\REopt\8hDDW\src\core\electric_utility.jl:411
┌ REopt | Warn: Your site location (10.80437, 106.71927) is more than 5 miles from the nearest AVERT region. Cannot calc

Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 105135 rows; 61339 cols; 315171 nonzeros; 1 integer variables (1 binary)
Coefficient ranges:
  Matrix  [1e-05, 1e+06]
  Cost    [1e-04, 2e+05]
  Bound   [1e+00, 1e+00]
  RHS     [1e+01, 1e+09]
Presolving model
91878 rows, 43609 cols, 261771 nonzeros  0s
76137 rows, 32351 cols, 251393 nonzeros  0s
73766 rows, 29980 cols, 260388 nonzeros  1s
Presolve reductions: rows 73766(-31369); columns 29980(-31359); nonzeros 260388(-54783) 

Solving MIP model with:
   73766 rows
   29980 cols (0 binary, 0 integer, 0 implied int., 29980 continuous, 0 domain fixed)
   260388 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p =>

┌ REopt | Info: REopt solved with 
│   termination_status(m) = OPTIMAL
└ @ REopt C:\Users\hauti\.julia\packages\REopt\8hDDW\src\core\reopt.jl:610
┌ REopt | Info: Solving took 18.432 seconds.
└ @ REopt C:\Users\hauti\.julia\packages\REopt\8hDDW\src\core\reopt.jl:611
┌ REopt | Info: Results processing took 28.076 seconds.
└ @ REopt C:\Users\hauti\.julia\packages\REopt\8hDDW\src\core\reopt.jl:616


Dict{String, Any} with 10 entries:
  "ElectricStorage" => Dict{String, Any}("size_kw"=>45.91, "size_kwh"=>114.38, …
  "Messages"        => Dict{Any, Any}("errors"=>Any[], "warnings"=>Any[("core_e…
  "ElectricUtility" => Dict{String, Any}("annual_renewable_electricity_supplied…
  "ElectricTariff"  => Dict{String, Any}("year_one_energy_cost_before_tax"=>634…
  "status"          => "optimal"
  "Site"            => Dict{String, Any}("annual_emissions_from_fuelburn_tonnes…
  "ElectricLoad"    => Dict{String, Any}("annual_peak_kw"=>199.052, "load_serie…
  "PV"              => Dict{String, Any}("electric_to_grid_series_kw"=>[0.0, 0.…
  "Financial"       => Dict("year_one_fuel_cost_before_tax"=>0.0, "lcc"=>1.2241…
  "solver_seconds"  => 18.432

In [11]:
#write_to_file(m, "reopt_model.lp")

In [78]:
using JSON

# Assuming 'results' is the output from run_reopt(m, p)
open("reopt_results.json", "w") do f
    JSON.print(f, results, 4) # The '4' adds indentation for readability
end

In [35]:
results["ElectricStorage"]

Dict{String, Any} with 5 entries:
  "size_kw"                   => 9.26
  "size_kwh"                  => 24.34
  "initial_capital_cost"      => 2.37237e5
  "soc_series_fraction"       => [0.861, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0…
  "storage_to_load_series_kw" => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, …

In [26]:
results["PV"][1]

Dict{String, Any} with 14 entries:
  "electric_to_grid_series_kw"    => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0…
  "production_factor_series"      => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0…
  "electric_to_load_series_kw"    => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0…
  "size_kw"                       => 22.0
  "name"                          => "Helsinki_Roof_PV"
  "annual_energy_produced_kwh"    => 17729.1
  "year_one_energy_produced_kwh"  => 18606.0
  "annual_energy_exported_kwh"    => 0.0
  "installed_cost_per_kw"         => 1500.0
  "electric_to_storage_series_kw" => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0…
  "electric_curtailed_series_kw"  => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0…
  "lifecycle_om_cost_after_tax"   => 3961.0
  "lcoe_per_kwh"                  => 0.0934
  "om_cost_per_kw"                => 15.0

In [29]:
results["PV"][1]["electric_to_storage_series_kw"]

8760-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 ⋮
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [38]:
results["Financial"]

Dict{String, Float64} with 31 entries:
  "year_one_fuel_cost_before_tax"                  => 0.0
  "lcc"                                            => 386629.0
  "lifecycle_om_costs_after_tax"                   => 75143.0
  "year_one_om_costs_before_tax"                   => 6261.02
  "lifecycle_capital_costs_plus_om_after_tax"      => 212858.0
  "year_one_total_operating_cost_after_tax"        => 16358.4
  "lifecycle_generation_tech_capital_costs"        => 16816.8
  "lifecycle_offgrid_other_annual_costs_after_tax" => 0.0
  "year_one_chp_standby_cost_after_tax"            => 0.0
  "replacements_present_cost_after_tax"            => 0.0
  "lifecycle_emissions_cost_health"                => 0.0
  "lifecycle_outage_cost"                          => 0.0
  "initial_capital_costs"                          => 2.70241e5
  "initial_capital_costs_after_incentives"         => 1.37715e5
  "lifecycle_MG_upgrade_and_fuel_cost"             => 0.0
  "lifecycle_elecbill_after_tax"                   =>

In [40]:
for (key, value) in results["Financial"]
    println("$key => $value")
end

year_one_fuel_cost_before_tax => 0.0
lcc => 386629.0118
lifecycle_om_costs_after_tax => 75143.0381
year_one_om_costs_before_tax => 6261.0188
lifecycle_capital_costs_plus_om_after_tax => 212857.9695
year_one_total_operating_cost_after_tax => 16358.387299999999
lifecycle_generation_tech_capital_costs => 16816.8308
lifecycle_offgrid_other_annual_costs_after_tax => 0.0
year_one_chp_standby_cost_after_tax => 0.0
replacements_present_cost_after_tax => 0.0
lifecycle_emissions_cost_health => 0.0
lifecycle_outage_cost => 0.0
initial_capital_costs => 270240.751
initial_capital_costs_after_incentives => 137714.9314
lifecycle_MG_upgrade_and_fuel_cost => 0.0
lifecycle_elecbill_after_tax => 173771.0423
lifecycle_storage_capital_costs => 120898.1006
lifecycle_om_costs_before_tax => 101544.646
lifecycle_emissions_cost_climate => 3469.93
lifecycle_fuel_costs_after_tax => 0.0
lifecycle_capital_costs => 137714.9314
lifecycle_production_incentive_after_tax => 0.0
year_one_fuel_cost_after_tax => 0.0
lifecy